In [7]:
import pandas as pd
import re
from datetime import datetime
from dateutil import parser
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar
from datetime import datetime, timedelta
import seaborn as sns
import numpy as np
from scipy.optimize import minimize_scalar


In [2]:
# Reference time for all power measurements
reference_time = datetime(1970, 1, 1, 0, 0, 0)



In [3]:


def parse_log_with_event_type_precise_ms(log_content):
    """
    Parse the Selenium log file and prioritize explicit durations in milliseconds when available.
    """
    events = []
    current_event = {}
    current_action = None

    for line in log_content:
        # Detect Fibonacci execution
        if "fibonacci" in line.lower():
            current_event['event_type'] = "Fibonacci"
            if "start" in line.lower():
                match = re.search(r"start.*\[(.*)\]", line)
                if match:
                    current_event['start_time'] = parser.parse(match.group(1))
            elif "end" in line.lower():
                match = re.search(r"end.*\[(.*)\]", line)
                if match:
                    current_event['end_time'] = parser.parse(match.group(1))
                    # Calculate duration in ms if no explicit duration is provided
                    if 'start_time' in current_event and 'end_time' in current_event and 'duration' not in current_event:
                        duration = (current_event['end_time'] - current_event['start_time']).total_seconds() * 1000
                        current_event['duration'] = int(duration)
                        # print(f"Parsed Fibonacci duration: {current_event['duration']} ms (calculated)")  # Debugging
                        events.append(current_event)
                        current_event = {}

        # Detect action runs
        # Detect action file path
        elif re.search(r"^\./", line):
            # Match action file paths with possible wildcards
            match = re.search(r"\./([^;]+)", line, re.IGNORECASE)
            if match:
                current_action = match.group(1).strip()
                

        elif "start test" in line.lower():
            match = re.search(r"start.*\[(.*)\]", line)
            if match:
                current_event = {'event_type': current_action}
                current_event['start_time'] = parser.parse(match.group(1))

        elif "end test" in line.lower():
            match = re.search(r"end.*\[(.*)\]", line)
            if match:
                current_event['end_time'] = parser.parse(match.group(1))
                # Calculate duration in ms if no explicit duration is provided
                if 'start_time' in current_event and 'end_time' in current_event and 'duration' not in current_event:
                    duration = (current_event['end_time'] - current_event['start_time']).total_seconds() * 1000
                    current_event['duration'] = int(duration)
                    # print(f"Parsed Action duration: {current_event['duration']} ms (calculated)")  # Debugging

        elif "duration of test" in line.lower():
            match = re.search(r"duration of test (\d+)", line)
            if match:
                # Use explicit duration in milliseconds
                current_event['duration'] = int(match.group(1))
                # print(f"Parsed explicit duration: {current_event['duration']} ms (explicit)")  # Debugging

        # Append the current event if fully parsed
        if 'start_time' in current_event and 'end_time' in current_event and 'duration' in current_event:
            events.append(current_event)
            current_event = {}

    return pd.DataFrame(events)




def construct_event_timeline_with_types_ms(config, log_events, default_fib_duration_ms=2000):
    """
    Construct a timeline of events using the `event_type` and iterations, saving durations in ms.
    """
    timeline = []
    current_time = 0  # Time in ms
    duration = 0
    # Script start
    timeline.append({"event": "script start", "event_type": None, "eventinit": None, "iteration": None, "time stamp (ms)": current_time, "duration (ms)": duration})

    # Handle Fibonacci events
    # fib_event = log_events[log_events['event_type'] == "Fibonacci"]
    # if not fib_event.empty:
    #    fib_start_time = current_time
    #    fib_duration = fib_event.iloc[0]['duration'] if 'duration' in fib_event.iloc[0] else default_fib_duration_ms
    #
    #    # Fibonacci start
    #    timeline.append({
    #        "event": "Fibonacci start",
    #        "event_type": "Fibonacci",
    #        "iteration": None,
    #        "duration (ms)": fib_start_time
    #    })
    #
    #    # Fibonacci end
    #    fib_end_time = fib_start_time + fib_duration
    #    timeline.append({
    #        "event": "Fibonacci end",
    #        "event_type": "Fibonacci",
    #        "iteration": None,
    #        "duration (ms)": fib_end_time
    #    })
    #
    #    current_time = fib_end_time

    # Waiting after Fibonacci
    #current_time += config['time_after_fib_before_run'] * 1000  # Convert to ms

    # Iterations over actions
    for action_index, action_name in enumerate(config['action order'], start=1):
        for iteration in range(1, config['repetitions in an iteration'] + 1):
            iteration_start_event = f"{action_name}, iteration {iteration}, start"
            iteration_end_event = f"{action_name}, iteration {iteration}, end"

            # Calculate index for the current iteration
            iteration_index = (action_index - 1) * config['repetitions in an iteration'] + (iteration - 1)+1
            if iteration_index >= len(log_events):
                raise ValueError(
                    f"Insufficient log entries for action {action_name}, iteration {iteration}. "
                    f"Expected index {iteration_index}, but log_events has {len(log_events)} rows."
                )

            # Fetch duration for the current iteration
            iteration_duration = log_events.iloc[iteration_index]['duration']
            #print(f"Action: {action_name}, Iteration: {iteration}, Duration: {iteration_duration} ms")  # Debugging

            # Add start event
            timeline.append({
                "event": iteration_start_event,
                "event_type": log_events.iloc[iteration_index]['event_type'],
                "eventinit": 'start',
                "iteration": iteration,
                "time stamp (ms)": current_time,
                "duration (ms)": iteration_duration
            })

            # Add duration
            current_time += iteration_duration

            # Add end event
            timeline.append({
                "event": iteration_end_event,
                "event_type": log_events.iloc[iteration_index]['event_type'],
                "eventinit": 'end',
                "iteration": iteration,
                "time stamp (ms)": current_time,
                "duration (ms)": 0
            })

            # Waiting between iterations
            if iteration < config['repetitions in an iteration']:
                current_time += config['waiting_between_iterations'] * 1000  # Convert to ms

        # Waiting between runs
        if action_index < len(config['action order']):
            current_time += (config['waiting_between_iterations']+config['waiting_between_runs']) * 1000  # Convert to ms

    return pd.DataFrame(timeline)


In [4]:
# Example usage
def extract_time_line(dict_):
    # Example configurations
    

    # Load log file (replace with actual file path)
    log_file_path = dict_['logpath']
    with open(log_file_path, 'r') as f:
        log_content = f.readlines()

    # Parse log file
    log_events = parse_log_with_event_type_precise_ms(log_content)
    #display(log_events)
    
    
    # Construct timeline
    timeline = construct_event_timeline_with_types_ms(dict_, log_events)

    # Define reference start time as 1970-01-01 00:00:00	

    reference_time = datetime(1970, 1, 1, 0, 0, 0)
        
    # Add a new column for datetime
    timeline['dateTime'] = timeline['duration (ms)'].apply(lambda ms: reference_time + timedelta(milliseconds=ms))

    # display(timeline.head())
    return log_events, timeline

# Run the program
#if __name__ == "__main__":
#    main()
# selenium dict test
# dict_ = all_dicts[8]
    
# Construct timeline for this framework (replace with actual timeline generation code)
# timeline = extract_time_line(dict_)

# print(timeline.head())


# Power consumption

In [5]:
# 4 = selenium dict test
# 2 = puppeteer
#dict_ = all_dicts[4]
#print(dict_['name'])
#print(dict_['logpath'])
# Construct timeline for this framework (replace with actual timeline generation code)
#logevent, timeline = extract_time_line(dict_)

#display(logevent)
#display(timeline)

In [6]:
# for dict_ in all_dicts:
#    print(dict_)